In [63]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# =========================
# 1. Load data
# =========================
df = pd.read_csv("hasil_sentimen.predictions - negative.csv")
df["clean_textdisplay"] = df["clean_textdisplay"].astype(str).apply(clean_text)

docs = df["clean_textdisplay"].fillna("").astype(str).tolist()

In [64]:
import re

# =========================
# 1. CUSTOM STOPWORDS (ISI SENDIRI)
# =========================
custom_stopwords = set([
    # isi sendiri nanti
    "yang",
    "dan",
    "di",
    "ke",
    "dari",
    "ini",
    "itu",
    "gak",
    "ga",
    "saya",
    "kamu",
    "dia",
    "kami",
    "mereka",
    "ya",
    "ad",
    "jual",
    "dan",
    "dari",
    "yg",
    "byk",
    "query",
    "nya",
])

# =========================
# 2. CLEAN FUNCTION
# =========================
def clean_text(text):
    text = str(text).lower()

    # hapus url
    text = re.sub(r"http\S+|www\S+", "", text)

    # hapus mention & hashtag
    text = re.sub(r"@\w+|#\w+", "", text)

    # hapus angka & simbol
    text = re.sub(r"[^a-z\s]", " ", text)

    # rapihin spasi
    text = re.sub(r"\s+", " ", text).strip()

    # hapus stopwords custom
    words = text.split()
    words = [w for w in words if w not in custom_stopwords]

    return " ".join(words)

In [65]:
# =========================
# 2. SOTA EMBEDDING MODEL
# =========================
model = SentenceTransformer("intfloat/multilingual-e5-large")

# E5 BUTUH PREFIX (IMPORTANT!)
embeddings = model.encode(
    ["query: " + d for d in docs],   # hanya untuk embedding
    show_progress_bar=True
)


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [66]:
# =========================
# 3. REDUCE DIMENSION (UMAP)
# =========================
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

In [67]:
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric="euclidean",
    cluster_selection_method="eom"
)

In [68]:
# =========================
# 5. BERTopic MODEL
# =========================
topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

2026-06-06 16:26:43,891 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-06 16:26:44,638 - BERTopic - Dimensionality - Completed ✓
2026-06-06 16:26:44,638 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-06 16:26:44,649 - BERTopic - Cluster - Completed ✓
2026-06-06 16:26:44,650 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-06 16:26:44,664 - BERTopic - Representation - Completed ✓


In [69]:
# =========================
# 6. TRAIN MODEL
# =========================
topics, probs = topic_model.fit_transform(docs)

2026-06-06 16:26:44,679 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

2026-06-06 16:26:55,473 - BERTopic - Embedding - Completed ✓
2026-06-06 16:26:55,474 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-06 16:26:56,119 - BERTopic - Dimensionality - Completed ✓
2026-06-06 16:26:56,119 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-06 16:26:56,125 - BERTopic - Cluster - Completed ✓
2026-06-06 16:26:56,126 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-06 16:26:56,138 - BERTopic - Representation - Completed ✓


In [70]:
topic_model.visualize_topics()

In [71]:
topic_model.visualize_barchart(top_n_topics=10)

In [72]:
# =========================
# 8. TOPIC SUMMARY
# =========================
print(topic_model.get_topic_info())


   Topic  Count                               Name  \
0     -1     72  -1_mobil_listrik_pajak_pemerintah   
1      0    160      0_vehicle_electric_mobil_beli   
2      1    130                 1_aja_pake_kita_lu   
3      2    104     2_indonesia_mobil_china_jepang   
4      3     72   3_baterai_mobil_vehicle_electric   
5      4     53             4_beli_mobil_harga_mau   
6      5     44        5_bensin_minyak_bakar_mobil   
7      6     16         6_atto_mobil_ngeselin_aneh   

                                      Representation  \
0  [mobil, listrik, pajak, pemerintah, tapi, vehi...   
1  [vehicle, electric, mobil, beli, listrik, harg...   
2  [aja, pake, kita, lu, takut, ada, kayak, bener...   
3  [indonesia, mobil, china, jepang, banyak, list...   
4  [baterai, mobil, vehicle, electric, listrik, t...   
5  [beli, mobil, harga, mau, masih, rugi, kalo, m...   
6  [bensin, minyak, bakar, mobil, polusi, bahan, ...   
7  [atto, mobil, ngeselin, aneh, gw, ada, fiesta,...   

        